# Deep Retention — Análisis financiero de retención

Este notebook integra los artefactos ya entrenados con el motor financiero. No reentrena el MLP, no usa `Exited` para decidir y no modifica los archivos fuente.

La probabilidad corregida se combina con `AnnualNetMargin_USD_Sim` mediante:

\[
GNE=(P_f 	imes E_i 	imes M_c)-C_i
\]

Los costos, efectividades y reglas comerciales del catálogo son **supuestos académicos**, no resultados causales ni condiciones bancarias reales.

## 1. Carga de datos y componentes

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    root
    for root in [Path.cwd(), *Path.cwd().parents]
    if (root / "data" / "processed" / "clientes_limpios.csv").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.financial import (
    calcular_umbrales_comerciales,
    evaluar_cartera,
    resumir_cartera,
    validar_catalogo,
)
from src.prediction import PredictorFuga

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clientes_limpios.csv"
CATALOG_PATH = PROJECT_ROOT / "data" / "incentivos.csv"

clientes = pd.read_csv(DATA_PATH)
catalogo = pd.read_csv(CATALOG_PATH)

print("Clientes cargados:", len(clientes))
print("Columnas:", clientes.shape[1])
print("Catálogo:", CATALOG_PATH.relative_to(PROJECT_ROOT).as_posix())

Clientes cargados: 10000
Columnas: 21
Catálogo: data/incentivos.csv


## 2. Consistencia financiera y caracterización

`AnnualNetMargin_USD_Sim` es el único margen utilizado en la GNE. Las comisiones y el margen financiero no se vuelven a sumar durante la decisión, porque ya forman parte del margen neto.

`Balance` se usa solo mediante percentiles internos: su unidad monetaria no está declarada y no se convierte a USD.

In [2]:
margen_recalculado = (
    clientes["AnnualCommissionIncome_USD_Sim"]
    + clientes["AnnualFinancialMargin_USD_Sim"]
    - clientes["AnnualOperatingCosts_USD_Sim"]
)
diferencia_margen = (
    margen_recalculado - clientes["AnnualNetMargin_USD_Sim"]
).abs()

print("Diferencia máxima del margen (USD):", round(diferencia_margen.max(), 4))
print(
    "Registros consistentes con tolerancia de 0,02 USD:",
    int(diferencia_margen.le(0.02).sum()),
)
print(
    "Clientes con margen no positivo:",
    int(clientes["AnnualNetMargin_USD_Sim"].le(0).sum()),
)

variables_perfil = [
    "Balance",
    "AnnualCommissionIncome_USD_Sim",
    "MonthlyTransactions_Sim",
    "MonthlyTransactionVolume_USD_Sim",
    "AnnualNetMargin_USD_Sim",
]
display(clientes[variables_perfil].describe().T)

Diferencia máxima del margen (USD): 0.0
Registros consistentes con tolerancia de 0,02 USD: 10000
Clientes con margen no positivo: 3602


,count,mean,std,min,25%,50%,75%,max
Balance,10000.0,76485.889288,62397.405202,0.00,0.0000,97198.540,127644.2400,250898.09
AnnualCommissionIncome_USD_Sim,10000.0,118.123944,50.826645,20.28,79.2000,111.600,151.9200,316.08
MonthlyTransactions_Sim,10000.0,19.653600,8.525824,1.00,13.0000,20.000,27.0000,44.00
MonthlyTransactionVolume_USD_Sim,10000.0,5276.529582,3534.181177,2.55,2671.2725,4500.660,7170.1225,19395.11
AnnualNetMargin_USD_Sim,10000.0,296.412185,323.928539,-163.62,-74.4750,362.985,551.5925,1253.97


## 3. Catálogo y umbrales comerciales

Las condiciones se evalúan antes de comparar la GNE:

- bonificación: actividad transaccional desde el percentil 50;
- exoneración: comisiones anuales desde el percentil 50;
- tasa preferencial: `Balance` desde el percentil 75 y volumen mensual desde el percentil 50;
- gestor personalizado: `Balance` desde el percentil 90 y volumen mensual desde el percentil 75.

Los mínimos de margen, costos y efectividades también son supuestos académicos versionados en el catálogo.

In [3]:
catalogo_validado = validar_catalogo(catalogo)
umbrales = calcular_umbrales_comerciales(clientes, catalogo_validado)

filas_umbrales = []
for regla, valores in umbrales.items():
    for percentil, valor in valores.items():
        filas_umbrales.append({
            "regla": regla,
            "percentil": percentil,
            "umbral_observado": valor,
        })

display(catalogo_validado)
display(pd.DataFrame(filas_umbrales))

,costo_anual_usd,efectividad,incentivo_id,margen_minimo_usd,nombre,percentil_balance_minimo,percentil_comision_minimo,percentil_transacciones_minimo,percentil_volumen_minimo,perfil_requerido
0,20,0.20,1,0,Bonificación transaccional,0.00,0.0,0.5,0.00,Transaccional
1,50,0.40,2,150,Exoneración de comisiones,0.00,0.5,0.0,0.00,Generador de comisiones
2,100,0.55,3,400,Tasa preferencial,0.75,0.0,0.0,0.50,Preferencial
3,200,0.75,4,800,Gestor personalizado,0.90,0.0,0.0,0.75,Patrimonial


,regla,percentil,umbral_observado
0,percentil_transacciones_minimo,0.50,20.0000
1,percentil_comision_minimo,0.50,111.6000
2,percentil_balance_minimo,0.75,127644.2400
3,percentil_balance_minimo,0.90,149244.7920
4,percentil_volumen_minimo,0.50,4500.6600
5,percentil_volumen_minimo,0.75,7170.1225


## 4. Predicción ajustada por lotes

La corrección por pesos de clase es aproximada: revierte el cambio de prior inducido por `class_weight`, pero no equivale a una calibración aprendida con datos independientes.

In [4]:
predictor = PredictorFuga()
probabilidades = predictor.predecir_lote(clientes)

probabilidades_individuales = np.array([
    predictor.predecir(clientes.iloc[[position]])
    for position in range(10)
])

print("Artefacto compatible:", predictor.artifact_id)
print("Probabilidades generadas:", len(probabilidades))
print("Mínimo:", round(float(probabilidades.min()), 6))
print("Máximo:", round(float(probabilidades.max()), 6))
print("Promedio:", round(float(probabilidades.mean()), 6))
print(
    "Equivalencia individual/lote (primeros 10):",
    bool(np.allclose(probabilidades_individuales, probabilidades[:10], rtol=1e-6, atol=1e-7)),
)

Artefacto compatible: 3a2890fbab71e9c8
Probabilidades generadas: 10000
Mínimo: 2.8e-05
Máximo: 1.0
Promedio: 0.195271
Equivalencia individual/lote (primeros 10): True


## 5. Decisiones consolidadas de la cartera

In [5]:
resultados = evaluar_cartera(
    clientes,
    probabilidades,
    catalogo_validado,
    umbrales,
)
resumen = resumir_cartera(resultados)

resumen_df = pd.DataFrame({
    "Indicador": list(resumen),
    "Valor": list(resumen.values()),
})
distribucion = (
    resultados.loc[resultados["decision"].eq("Retener"), "incentivo"]
    .value_counts()
    .rename_axis("Incentivo")
    .reset_index(name="Clientes")
)

deficitarios_retenidos = (
    resultados["decision"].eq("Retener")
    & resultados["margen_anual_usd"].le(0)
).sum()

print("Clientes deficitarios con gasto:", int(deficitarios_retenidos))
print(
    "Todas las retenciones tienen GNE positiva:",
    bool(resultados.loc[resultados["decision"].eq("Retener"), "gne"].gt(0).all()),
)
display(resumen_df)
display(distribucion)
display(
    resultados.drop(columns="alternativas").head(10)
)

Clientes deficitarios con gasto: 0
Todas las retenciones tienen GNE positiva: True


,Indicador,Valor
0,clientes_analizados,10000.000000
1,clientes_retener,1214.000000
2,clientes_no_intervencion,5184.000000
3,clientes_salida_permitida,3602.000000
4,inversion_total_usd,66640.000000
5,valor_rescatable_estimado_usd,156739.685817
6,gne_total_usd,90099.685817
7,roi_proyectado_pct,135.203610


,Incentivo,Clientes
0,Exoneración de comisiones,536
1,Bonificación transaccional,367
2,Tasa preferencial,297
3,Gestor personalizado,14


,CustomerId,MonthlyTransactions_Sim,AnnualCommissionIncome_USD_Sim,Balance,MonthlyTransactionVolume_USD_Sim,probabilidad_fuga,margen_anual_usd,decision,motivo,incentivo_id,incentivo,perfil_requerido,efectividad,costo,gne,elegible
0,15634602,19,93.24,0.00,4195.46,0.238236,-98.98,Salida permitida,Cliente con margen neto anual no positivo; se ...,NaN,None,None,0.00,0.0,0.000000,False
1,15647311,26,120.96,83807.86,5335.43,0.137945,341.94,No intervención,Cliente rentable sin incentivos elegibles con ...,NaN,None,None,0.00,0.0,0.000000,False
2,15619304,21,117.00,159660.80,4617.19,0.998095,558.31,Retener,Incentivo elegible con la mayor GNE positiva.,3.0,Tasa preferencial,Preferencial,0.55,100.0,206.485670,True
3,15701354,15,82.80,0.00,1747.07,0.030752,-111.89,Salida permitida,Cliente con margen neto anual no positivo; se ...,NaN,None,None,0.00,0.0,0.000000,False
4,15737888,29,118.92,125510.82,5681.50,0.008252,606.17,No intervención,Cliente rentable sin incentivos elegibles con ...,NaN,None,None,0.00,0.0,0.000000,False
5,15574012,14,99.84,113755.78,4126.60,0.265480,613.52,No intervención,Cliente rentable sin incentivos elegibles con ...,NaN,None,None,0.00,0.0,0.000000,False
6,15592531,34,138.00,0.00,412.96,0.046223,-111.03,Salida permitida,Cliente con margen neto anual no positivo; se ...,NaN,None,None,0.00,0.0,0.000000,False
7,15656148,30,201.60,115046.74,3186.43,0.999998,352.51,Retener,Incentivo elegible con la mayor GNE positiva.,2.0,Exoneración de comisiones,Generador de comisiones,0.40,50.0,91.003706,True
8,15792365,32,166.56,142051.07,7863.99,0.125792,675.37,No intervención,Cliente rentable sin incentivos elegibles con ...,NaN,None,None,0.00,0.0,0.000000,False
9,15592389,29,139.80,134603.88,3215.19,0.015737,429.93,No intervención,Cliente rentable sin incentivos elegibles con ...,NaN,None,None,0.00,0.0,0.000000,False


## 6. Comparación con la primera implementación

In [6]:
referencia_anterior = {
    "clientes_retener": 2319,
    "gne_total_usd": 161651.15,
}

comparacion = pd.DataFrame({
    "Indicador": ["Clientes a retener", "GNE total proyectada (USD)"],
    "Primera implementación": [
        referencia_anterior["clientes_retener"],
        referencia_anterior["gne_total_usd"],
    ],
    "Reglas comerciales nuevas": [
        resumen["clientes_retener"],
        round(resumen["gne_total_usd"], 2),
    ],
})
comparacion["Diferencia"] = (
    comparacion["Reglas comerciales nuevas"]
    - comparacion["Primera implementación"]
)
display(comparacion)

,Indicador,Primera implementación,Reglas comerciales nuevas,Diferencia
0,Clientes a retener,2319.00,1214.00,-1105.00
1,GNE total proyectada (USD),161651.15,90099.69,-71551.46


## 7. Conclusiones

- Se analizaron 10.000 clientes y los 10.000 márgenes cumplen la identidad financiera con tolerancia de 0,02 USD.
- Las nuevas condiciones comerciales recomiendan **retener a 1.214 clientes**: 367 con bonificación transaccional, 536 con exoneración de comisiones, 297 con tasa preferencial y 14 con gestor personalizado.
- **3.602 clientes** tienen margen no positivo y se clasifican como `Salida permitida`; ninguno recibe gasto. Otros **5.184 clientes rentables** quedan en `No intervención` porque no existe una alternativa simultáneamente elegible y con GNE positiva.
- La inversión académica proyectada es **USD 66.640,00**, el valor rescatable estimado es **USD 156.739,69** y la GNE total es **USD 90.099,69**.
- Frente a la primera implementación, se recomiendan 1.105 intervenciones menos y la GNE proyectada disminuye USD 71.551,46. La diferencia se debe al filtrado por actividad transaccional, comisiones, volumen y percentiles relativos de `Balance`; no se forzaron los resultados anteriores.
- Estas cifras son una simulación académica sobre una cartera que incluye registros usados durante el entrenamiento. No representan ganancias realizadas ni una evaluación predictiva independiente.